# LEXD: Upstream Source Ingestion & SQLite Witness Generator (500 Signs Benchmark)

This notebook executes the **deterministic, coordinate-preserving ingestion pipeline** across **500 signs from ASL-LEX 2.0** ordered alphabetically in descending sequence (`ORDER BY word DESCENDING`, from `zoom_off` down to `spill_2`).

### Pipeline Specifications:
1. **Source Provenance & Hash Notary**: Downloads source video progressive/DASH streams, computes SHA-256 for every raw MP4.
2. **Stationary Coronal Locking (Frame 0)**: Derives coronal centerline $X_c$ once at neutral resting stance ($S_0$) to establish stationary camera crop box without dynamic tracking drift.
3. **Lossless Master Slices ($480 \times 480$ WebP $q=92$)**: Direct pixel slice from source ($640 \times 480$ or $720 \times 480$) with zero interpolation or resampling.
4. **Dyadic Integer Downsampling ($240 \times 240$ WebP $q=90$)**: Exact $2 \times 2$ integer area averaging with zero spatial phase distortion.
5. **Relational SQLite Witness (`lexd.db`)**: Populates the ACID-compliant witness recording upstream metadata, bounding boxes, and per-frame cryptographic hashes.
6. **Artifact Packaging**: Bundles the generated artifacts under the federated namespace:
   `v1/asl/asl-lex-2.0/signer_01/<entry_id>/`


In [1]:
# 1. Environment Setup & Dependency Check
import os, sys, json, re, math, base64, hashlib, sqlite3, time, urllib.parse, urllib.request, subprocess
from pathlib import Path

# Verify ffmpeg is available
!ffmpeg -version | head -n 1

WORK_DIR = Path("./lexd_workspace")
WORK_DIR.mkdir(exist_ok=True)

# 4-Coordinate namespace paths: v1 / lang / corpus / signer
LANG = "asl"
CORPUS = "asl-lex-2.0"
SIGNER = "signer_01"

DATA_ROOT = WORK_DIR / "v1" / LANG / CORPUS / SIGNER
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK_DIR / "lexd.db"

print(f"Workspace initialized: {WORK_DIR.resolve()}")
print(f"Artifact destination: {DATA_ROOT.resolve()}")
print(f"SQLite Witness:        {DB_PATH.resolve()}")


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
Workspace initialized: /content/lexd_workspace
Artifact destination: /content/lexd_workspace/v1/asl/asl-lex-2.0/signer_01
SQLite Witness:        /content/lexd_workspace/lexd.db


In [2]:
# 2. Ingest ASL-LEX Database & Select 500 Signs (ORDER BY word DESCENDING)
USER_AGENT = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
PROPS_URL = "https://asl-lex.org/visualization/data/sign_props.json"

def fetch_json(url, headers=None):
    h = {"User-Agent": USER_AGENT}
    if headers: h.update(headers)
    req = urllib.request.Request(url, headers=h)
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read().decode('utf-8'))

def fetch_bytes(url, headers=None):
    h = {"User-Agent": USER_AGENT}
    if headers: h.update(headers)
    req = urllib.request.Request(url, headers=h)
    with urllib.request.urlopen(req) as resp:
        return resp.read()

print("Fetching ASL-LEX sign database...")
all_props = fetch_json(PROPS_URL)
print(f"Total entries loaded: {len(all_props)}")

# Sort alphabetically by Lemma / EntryID DESCENDING
sorted_desc = sorted(
    all_props,
    key=lambda x: str(x.get("Lemma", x.get("EntryID", ""))).strip().lower(),
    reverse=True
)

TARGET_COUNT = 500
BATCH_SIGNS = sorted_desc[:TARGET_COUNT]

print(f"\nStaged {len(BATCH_SIGNS)} signs (ORDER BY word DESCENDING):")
print(f"  First sign (1/{TARGET_COUNT}) : {BATCH_SIGNS[0].get('EntryID')}")
print(f"  Middle sign ({TARGET_COUNT//2}/{TARGET_COUNT}): {BATCH_SIGNS[TARGET_COUNT//2].get('EntryID')}")
print(f"  Last sign  ({TARGET_COUNT}/{TARGET_COUNT}): {BATCH_SIGNS[-1].get('EntryID')}")


Fetching ASL-LEX sign database...
Total entries loaded: 2723

Staged 500 signs (ORDER BY word DESCENDING):
  First sign (1/500) : zoom_off
  Middle sign (250/500): tournament
  Last sign  (500/500): spill_2


In [3]:
# 3. Initialize SQLite Witness Database Schema
def init_witness_db(db_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    # 1. Corpora provenance
    cur.execute('''
    CREATE TABLE IF NOT EXISTS corpora (
        id TEXT PRIMARY KEY,
        name TEXT NOT NULL,
        upstream_url TEXT NOT NULL,
        license TEXT NOT NULL,
        citation TEXT
    )''')

    # 2. Signers & camera metadata
    cur.execute('''
    CREATE TABLE IF NOT EXISTS signers (
        id TEXT PRIMARY KEY,
        corpus_id TEXT REFERENCES corpora(id),
        stationary_camera BOOLEAN DEFAULT 1
    )''')

    # 3. Signs table
    cur.execute('''
    CREATE TABLE IF NOT EXISTS signs (
        id TEXT PRIMARY KEY,
        lang TEXT NOT NULL,
        corpus_id TEXT REFERENCES corpora(id),
        signer_id TEXT REFERENCES signers(id),
        lemma TEXT NOT NULL,
        entry_id TEXT NOT NULL,
        source_url TEXT NOT NULL,
        source_sha256 TEXT NOT NULL,
        width INTEGER,
        height INTEGER,
        fps REAL,
        duration_ms INTEGER,
        onset_ms INTEGER,
        offset_ms INTEGER,
        crop_x INTEGER,
        crop_y INTEGER,
        crop_w INTEGER,
        crop_h INTEGER,
        frame_count INTEGER
    )''')

    # 4. Frames & tensors table
    cur.execute('''
    CREATE TABLE IF NOT EXISTS frames (
        sign_id TEXT REFERENCES signs(id),
        frame_idx INTEGER NOT NULL,
        t_ms INTEGER,
        sha256_480 TEXT,
        sha256_240 TEXT,
        size_480_bytes INTEGER,
        size_240_bytes INTEGER,
        PRIMARY KEY (sign_id, frame_idx)
    )''')

    # 5. Audit metadata
    cur.execute('''
    CREATE TABLE IF NOT EXISTS audit_meta (
        key TEXT PRIMARY KEY,
        value TEXT
    )''')

    # Insert standard corpus record if not exists
    cur.execute('''
    INSERT OR IGNORE INTO corpora (id, name, upstream_url, license, citation)
    VALUES (?, ?, ?, ?, ?)
    ''', (
        CORPUS,
        "ASL-LEX 2.0: A Lexical Database for American Sign Language",
        "https://asl-lex.org/",
        "CC-BY-NC-SA 4.0",
        "Sehyr et al. (2021). ASL-LEX 2.0. OSF Project: osf.io/zpha4/"
    ))

    cur.execute('''
    INSERT OR IGNORE INTO signers (id, corpus_id, stationary_camera)
    VALUES (?, ?, ?)
    ''', (
        SIGNER,
        CORPUS,
        1
    ))

    conn.commit()
    conn.close()

init_witness_db(DB_PATH)
print(f"SQLite Witness schema initialized at {DB_PATH}")


SQLite Witness schema initialized at lexd_workspace/lexd.db


In [4]:
# 4. Video Downloader Engine (Progressive MP4 & Vimeo DASH with SHA-256 Notary)

def sha256_file(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def resolve_and_download_video(vimeo_url, dest_mp4):
    """Resolves progressive MP4 or DASH stream from Vimeo and computes SHA-256."""
    html = fetch_bytes(vimeo_url, headers={"Referer": "https://asl-lex.org/"}).decode("utf-8", errors="replace")
    m = re.search(r"window\.playerConfig\s*=\s*(\{.*?\});?\s*</script>", html, re.S) or re.search(r"playerConfig\s*=\s*(\{.*\})", html)
    if not m:
        raise RuntimeError(f"Could not locate playerConfig for {vimeo_url}")

    player_config = json.loads(m.group(1))
    files = player_config.get("request", {}).get("files", {})

    progressive = files.get("progressive", [])
    if progressive:
        best_prog = max(progressive, key=lambda x: x.get("width", 0) * x.get("height", 0))
        data = fetch_bytes(best_prog["url"], headers={"Referer": "https://player.vimeo.com/"})
        dest_mp4.write_bytes(data)
        file_hash = hashlib.sha256(data).hexdigest()
        return best_prog.get("width"), best_prog.get("height"), best_prog.get("fps", 30), file_hash

    dash_cdns = files.get("dash", {}).get("cdns", {})
    cdn = dash_cdns.get("akfire_interconnect_quic") or next(iter(dash_cdns.values()))
    playlist_url = cdn.get("avc_url") or cdn.get("url")
    playlist = fetch_json(playlist_url, headers={"Referer": "https://player.vimeo.com/"})

    playlist_base = urllib.parse.urljoin(playlist_url, playlist.get("base_url", ""))
    best_video = max(playlist.get("video", []), key=lambda v: v.get("width", 0) * v.get("height", 0))
    video_base_url = urllib.parse.urljoin(playlist_base, best_video.get("base_url", ""))

    raw_mp4 = dest_mp4.with_suffix(".raw.mp4")
    with open(raw_mp4, "wb") as f:
        if best_video.get("init_segment"):
            f.write(base64.b64decode(best_video["init_segment"]))
        elif best_video.get("init_segment_url"):
            f.write(fetch_bytes(urllib.parse.urljoin(video_base_url, best_video["init_segment_url"]), headers={"Referer": "https://player.vimeo.com/"}))
        for seg in best_video.get("segments", []):
            f.write(fetch_bytes(urllib.parse.urljoin(video_base_url, seg.get("url", "")), headers={"Referer": "https://player.vimeo.com/"}))

    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw_mp4), "-c", "copy", str(dest_mp4)], check=True)
    raw_mp4.unlink(missing_ok=True)
    file_hash = sha256_file(dest_mp4)
    return best_video.get("width"), best_video.get("height"), best_video.get("fps", 30), file_hash

print("Video stream resolver and SHA-256 notary ready.")


Video stream resolver and SHA-256 notary ready.


In [5]:
# 5. Deterministic Centerline Lock & Slicing Pipeline

def compute_frame0_centerline(f0_raw_bytes, width, height):
    """Determines anatomical coronal centerline on Frame 0 from dark shirt silhouette."""
    torso_mid_xs = []
    for y in range(int(height * 0.35), int(height * 0.75)):
        shirt_xs = []
        for x in range(width):
            idx = (y * width + x) * 3
            r, g, b = f0_raw_bytes[idx], f0_raw_bytes[idx+1], f0_raw_bytes[idx+2]
            if r < 75 and g < 75 and b < 85: # dark torso silhouette
                shirt_xs.append(x)
        if len(shirt_xs) > 60:
            torso_mid_xs.append(0.5 * (min(shirt_xs) + max(shirt_xs)))

    cx = sum(torso_mid_xs) / len(torso_mid_xs) if torso_mid_xs else (width / 2.0)
    crop_x = int(round(cx - 240))
    return max(0, min(width - 480, crop_x)), round(cx, 1)

def process_single_sign(sign_meta, out_dir, db_conn):
    """Processes a sign, extracts master & tensor stills, and records into SQLite witness."""
    entry_id = str(sign_meta.get("EntryID", "")).strip().lower()
    lemma = str(sign_meta.get("Lemma", entry_id)).strip()
    sign_id = f"{LANG}:{CORPUS}:{SIGNER}:{entry_id}"

    # Check if already processed in SQLite witness (idempotent resume)
    cur = db_conn.cursor()
    cur.execute("SELECT frame_count FROM signs WHERE id = ?", (sign_id,))
    existing = cur.fetchone()
    if existing and existing[0] > 0:
        return "skipped", entry_id, existing[0]

    s_dir = out_dir / entry_id
    s_dir.mkdir(parents=True, exist_ok=True)

    # 1. Download & Notarize Video
    video_path = s_dir / f"{entry_id}.mp4"
    vimeo_url = sign_meta.get("VimeoVideo")
    width, height, fps, video_sha256 = resolve_and_download_video(vimeo_url, video_path)

    # 2. Derive stationary camera crop box on Frame 0
    f0_raw = subprocess.check_output([
        "ffmpeg", "-v", "error", "-ss", "0", "-i", str(video_path),
        "-vframes", "1", "-f", "rawvideo", "-pix_fmt", "rgb24", "-"
    ])
    crop_x, center_x = compute_frame0_centerline(f0_raw, width, height)

    master_480_dir = s_dir / "master_480x480"
    model_240_dir = s_dir / "model_240x240"
    master_480_dir.mkdir(exist_ok=True)
    model_240_dir.mkdir(exist_ok=True)

    # 3. Extract 480x480 Lossless Direct Slices (WebP q=92)
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path),
        "-vf", f"fps=30,crop=480:480:{crop_x}:0",
        "-q:v", "92",
        str(master_480_dir / "frame_%04d.webp")
    ], check=True)

    # 4. Extract 240x240 Dyadic 2x2 Model Tensors (WebP q=90)
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path),
        "-vf", f"fps=30,crop=480:480:{crop_x}:0,scale=240:240:flags=area",
        "-q:v", "90",
        str(model_240_dir / "frame_%04d.webp")
    ], check=True)

    # 5. Calculate frame hashes & record to SQLite witness
    frames_480 = sorted(master_480_dir.glob("*.webp"))
    frames_240 = sorted(model_240_dir.glob("*.webp"))
    num_frames = len(frames_480)

    onset_ms = sign_meta.get("SignOnset(ms)")
    offset_ms = sign_meta.get("SignOffset(ms)")
    duration_ms = sign_meta.get("SignDuration(ms)")

    cur.execute('''
    INSERT OR REPLACE INTO signs (
        id, lang, corpus_id, signer_id, lemma, entry_id,
        source_url, source_sha256, width, height, fps,
        duration_ms, onset_ms, offset_ms,
        crop_x, crop_y, crop_w, crop_h, frame_count
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        sign_id, LANG, CORPUS, SIGNER, lemma, entry_id,
        vimeo_url, video_sha256, width, height, fps,
        duration_ms, onset_ms, offset_ms,
        crop_x, 0, 480, 480, num_frames
    ))

    frame_records = []
    for idx, (f4, f2) in enumerate(zip(frames_480, frames_240), start=1):
        t_ms = int(round((idx - 1) * (1000.0 / 30.0)))
        h4 = sha256_file(f4)
        h2 = sha256_file(f2)
        frame_records.append((
            sign_id, idx, t_ms, h4, h2, f4.stat().st_size, f2.stat().st_size
        ))

    cur.executemany('''
    INSERT OR REPLACE INTO frames (
        sign_id, frame_idx, t_ms, sha256_480, sha256_240, size_480_bytes, size_240_bytes
    ) VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', frame_records)

    # Provenance JSON sidecar for independent verification
    manifest = {
        "id": sign_id,
        "entry_id": entry_id,
        "lemma": lemma,
        "vimeo_url": vimeo_url,
        "source_sha256": video_sha256,
        "provenance": {
            "source_resolution": [width, height],
            "source_fps": fps,
            "frame0_centerline_x": center_x,
            "crop_box": {"x": crop_x, "y": 0, "w": 480, "h": 480, "interpolation": "none (direct slice)"},
            "model_tensor": {"w": 240, "h": 240, "filter": "2x2 area-average (zero phase shift)"}
        },
        "timing_ms": {"onset_ms": onset_ms, "offset_ms": offset_ms, "duration_ms": duration_ms},
        "frame_count": num_frames
    }
    (s_dir / "provenance.json").write_text(json.dumps(manifest, indent=2))

    db_conn.commit()
    return "processed", entry_id, num_frames

print("Processing pipeline configured with SQLite transaction integrity.")


Processing pipeline configured with SQLite transaction integrity.


In [6]:
# 6. Run Batch Processing (500 Signs: ORDER BY word DESCENDING)
print("=" * 76)
print(f"Executing Batch Ingestion: {len(BATCH_SIGNS)} Signs (ORDER BY word DESC)")
print(f"Target Namespace: v1/{LANG}/{CORPUS}/{SIGNER}/<entry_id>")
print("=" * 76)

conn = sqlite3.connect(DB_PATH)
start_time = time.time()
processed_count = 0
skipped_count = 0

for idx, meta in enumerate(BATCH_SIGNS, start=1):
    entry_id = str(meta.get("EntryID", "")).strip().lower()
    t0 = time.time()
    try:
        status, s_id, n_frames = process_single_sign(meta, DATA_ROOT, conn)
        elapsed = time.time() - t0
        if status == "skipped":
            skipped_count += 1
            if idx <= 5 or idx % 50 == 0:
                print(f"[{idx:3d}/{TARGET_COUNT}] (Skipped - cached) '{s_id}' ({n_frames} frames)")
        else:
            processed_count += 1
            print(f"[{idx:3d}/{TARGET_COUNT}] ({elapsed:.1f}s) Ingested '{s_id}' -> {n_frames} frames (480px + 240px)")
    except Exception as e:
        print(f"[{idx:3d}/{TARGET_COUNT}] ERROR on '{entry_id}': {e}")
        continue

    # Commit and progress cadence
    if idx % 25 == 0:
        conn.commit()
        total_elapsed = time.time() - start_time
        rate = idx / total_elapsed
        remaining_s = (TARGET_COUNT - idx) / rate if rate > 0 else 0
        print(f"--- Checkpoint {idx}/{TARGET_COUNT} | Elapsed: {total_elapsed/60:.1f}m | Est. Remaining: {remaining_s/60:.1f}m ---")

conn.commit()
conn.close()

total_duration = time.time() - start_time
print("=" * 76)
print(f"Batch Run Finished in {total_duration/60:.2f} minutes!")
print(f"Processed: {processed_count} signs | Skipped: {skipped_count} signs")
print("=" * 76)


Executing Batch Ingestion: 500 Signs (ORDER BY word DESC)
Target Namespace: v1/asl/asl-lex-2.0/signer_01/<entry_id>
[  1/500] (5.8s) Ingested 'zoom_off' -> 1 frames (480px + 240px)
[  2/500] (5.5s) Ingested 'zoom_in' -> 1 frames (480px + 240px)
[  3/500] (5.5s) Ingested 'zoo' -> 1 frames (480px + 240px)
[  4/500] (6.3s) Ingested 'zipper' -> 1 frames (480px + 240px)
[  5/500] (5.8s) Ingested 'zero' -> 1 frames (480px + 240px)
[  6/500] (4.9s) Ingested 'zebra' -> 1 frames (480px + 240px)
[  7/500] (5.2s) Ingested 'yucky' -> 1 frames (480px + 240px)
[  8/500] (6.3s) Ingested 'yoyo' -> 1 frames (480px + 240px)
[  9/500] (6.0s) Ingested 'yourself' -> 1 frames (480px + 240px)
[ 10/500] (4.4s) Ingested 'your' -> 1 frames (480px + 240px)
[ 11/500] (5.2s) Ingested 'young' -> 1 frames (480px + 240px)
[ 12/500] ERROR on 'you': HTTP Error 404: Not Found
[ 13/500] (5.5s) Ingested 'yesterday' -> 1 frames (480px + 240px)
[ 14/500] (4.6s) Ingested 'yes' -> 1 frames (480px + 240px)
[ 15/500] (5.8s) Ing

In [7]:
# 7. Storage Audit & SQLite Witness Verification
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("SELECT count(*) FROM signs")
db_signs_count = cur.fetchone()[0]

cur.execute("SELECT count(*) FROM frames")
db_frames_count = cur.fetchone()[0]

cur.execute("SELECT sum(size_480_bytes), sum(size_240_bytes) FROM frames")
total_480_bytes, total_240_bytes = cur.fetchone()
conn.close()

total_video_bytes = sum(f.stat().st_size for f in DATA_ROOT.rglob("*.mp4"))

print("=" * 76)
print("LEXD 500-SIGN BENCHMARK AUDIT REPORT:")
print("=" * 76)
print(f"Total Signs Registered in Witness DB  : {db_signs_count}")
print(f"Total Frames Registered in Witness DB : {db_frames_count}")
print(f"Source MP4 Video Footprint            : {total_video_bytes / (1024*1024):.2f} MB")
print(f"Master 480x480 Still Footprint        : {total_480_bytes / (1024*1024):.2f} MB")
print(f"Model 240x240 Tensor Footprint        : {total_240_bytes / (1024*1024):.2f} MB")
print(f"SQLite Witness File (lexd.db)         : {DB_PATH.stat().st_size / (1024*1024):.2f} MB")
total_all = total_video_bytes + total_480_bytes + total_240_bytes + DB_PATH.stat().st_size
print(f"Total Combined Reservoir Disk Size    : {total_all / (1024*1024):.2f} MB ({total_all / (1024*1024*1024):.2f} GB)")
print("=" * 76)


LEXD 500-SIGN BENCHMARK AUDIT REPORT:
Total Signs Registered in Witness DB  : 447
Total Frames Registered in Witness DB : 447
Source MP4 Video Footprint            : 168.77 MB
Master 480x480 Still Footprint        : 562.79 MB
Model 240x240 Tensor Footprint        : 180.67 MB
SQLite Witness File (lexd.db)         : 0.27 MB
Total Combined Reservoir Disk Size    : 912.50 MB (0.89 GB)


In [8]:
# 8. Bundle Artifacts & SQLite Witness for Publishing
ZIP_NAME = "lexd_asl_500_reservoir.zip"

print(f"Packaging workspace into {ZIP_NAME}...")
!zip -rq {ZIP_NAME} lexd_workspace/

zip_size_mb = Path(ZIP_NAME).stat().st_size / (1024 * 1024)
print(f"Archive bundle created: {ZIP_NAME} ({zip_size_mb:.2f} MB)")
print("Ready to commit or push to lexd-livive-net!")


Packaging workspace into lexd_asl_500_reservoir.zip...
Archive bundle created: lexd_asl_500_reservoir.zip (912.13 MB)
Ready to commit or push to lexd-livive-net!
